# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [1]:
# notebooks/02_limpieza_enriquecimiento.ipynb

import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Configuración de pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 1. CARGA DE DATOS
# Cargamos el dataset que dejaste listo tras el EDA
df_train = pd.read_csv('../data/interim/train_set.csv')
print(f"Filas originales: {df_train.shape[0]}")

Filas originales: 16512


In [2]:
def limpiar_anomalias(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica las reglas de negocio y correcciones de inconsistencias
    descubiertas en la fase de EDA.
    """
    df_clean = df.copy()
    
    # 1. Eliminar censura del target (Valores topados artificialmente)
    # Supuesto: Predecir sobre datos censurados sesgará la predicción superior del algoritmo.
    filtro_censura = df_clean['median_house_value'] < 500001.0
    
    # 2. Eliminar Inconsistencias lógicas ("casas fantasma")
    filtro_fantasmas = df_clean['population'] >= df_clean['households']
    
    # 3. Eliminar hacinamiento extremo (> 20 personas por hogar promedio)
    # Supuesto: Es un error de recolección del censo o un outlier extremo no representativo.
    filtro_hacinamiento = (df_clean['population'] / df_clean['households']) <= 20.0
    
    # Aplicar filtros
    mask_total = filtro_censura & filtro_fantasmas & filtro_hacinamiento
    df_clean = df_clean[mask_total].reset_index(drop=True)
    
    print(f"Filas tras limpieza lógica: {df_clean.shape[0]} (Retenidas: {df_clean.shape[0]/df.shape[0]:.2%})")
    return df_clean

# Ejecución
df_step1 = limpiar_anomalias(df_train)

Filas tras limpieza lógica: 15739 (Retenidas: 95.32%)


In [3]:
def imputar_faltantes(df: pd.DataFrame, n_neighbors: int = 5) -> pd.DataFrame:
    """
    Imputa variables numéricas utilizando el algoritmo de K-Nearest Neighbors.
    Excluye la variable categórica y el target para evitar data leakage.
    """
    df_imputed = df.copy()
    
    # Separar categóricas y target (NUNCA imputar usando el target en la industria para evitar leakage)
    target = df_imputed['median_house_value']
    categoricas = df_imputed[['ocean_proximity']]
    
    # Variables a usar para encontrar a los vecinos
    features_knn = df_imputed.drop(columns=['median_house_value', 'ocean_proximity'])
    column_names = features_knn.columns
    
    # Aplicar KNN Imputer
    # Supuesto: Vecindarios con métricas similares (cuartos, población, lat/lon) 
    # tendrán un número similar de dormitorios.
    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
    features_imputed = imputer.fit_transform(features_knn)
    
    # Reconstruir el DataFrame
    df_features = pd.DataFrame(features_imputed, columns=column_names, index=df_imputed.index)
    
    # Unir todo de nuevo
    df_final = pd.concat([df_features, categoricas, target], axis=1)
    
    nulos_restantes = df_final.isnull().sum().sum()
    print(f"Imputación KNN finalizada. Total de nulos en el dataset: {nulos_restantes}")
    
    return df_final

# Ejecución
df_step2 = imputar_faltantes(df_step1, n_neighbors=5)

Imputación KNN finalizada. Total de nulos en el dataset: 0


In [4]:
def calcular_distancia_haversine(lat1, lon1, lat2, lon2):
    """
    Calcula la distancia ortodrómica en kilómetros entre dos puntos
    en la Tierra usando sus latitudes y longitudes.
    """
    R = 6371.0 # Radio de la Tierra en km
    
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return R * c

def ingenieria_caracteristicas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crea nuevas variables matemáticas y espaciales para potenciar
    la capacidad predictiva del modelo.
    """
    df_feat = df.copy()
    
    # 1. RATIOS ESTRUCTURALES (Tus sugerencias)
    # Protegemos contra división por cero agregando un epsilon o validando
    df_feat['rooms_per_household'] = df_feat['total_rooms'] / df_feat['households']
    df_feat['bedrooms_per_room'] = df_feat['total_bedrooms'] / df_feat['total_rooms']
    df_feat['population_per_household'] = df_feat['population'] / df_feat['households']
    
    # 2. FEATURE ENGINEERING ESPACIAL (Coordenadas de LA y SF)
    LA_COORDS = (34.0522, -118.2437)
    SF_COORDS = (37.7749, -122.4194)
    
    df_feat['dist_to_LA'] = calcular_distancia_haversine(
        df_feat['latitude'], df_feat['longitude'], LA_COORDS[0], LA_COORDS[1]
    )
    df_feat['dist_to_SF'] = calcular_distancia_haversine(
        df_feat['latitude'], df_feat['longitude'], SF_COORDS[0], SF_COORDS[1]
    )
    
    # 3. INTERACCIONES AVANZADAS
    # Densidad de cuartos en la zona geográfica
    df_feat['rooms_per_population'] = df_feat['total_rooms'] / (df_feat['population'] + 1)
    
    print(f"Feature Engineering completado. Nuevas variables generadas: 6")
    return df_feat

# Ejecución
df_step3 = ingenieria_caracteristicas(df_step2)

Feature Engineering completado. Nuevas variables generadas: 6


In [6]:
def codificar_categoricas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica One-Hot Encoding evitando la multicolinealidad perfecta.
    Categoría base (omitida) será absorbida por el intercepto en modelos lineales.
    """
    df_encoded = df.copy()
    
    # CAMBIO CRÍTICO: drop_first=True
    df_encoded = pd.get_dummies(
        df_encoded, 
        columns=['ocean_proximity'], 
        prefix='ocean', 
        drop_first=True,  # Mandatory para LinearRegression y SGD
        dtype=int
    )
    
    columnas_nuevas = [col for col in df_encoded.columns if 'ocean_' in col]
    print(f"One-Hot Encoding. Columnas creadas (K-1 dummies): {columnas_nuevas}")
    
    return df_encoded

# Ejecución
df_step4 = codificar_categoricas(df_step3)

One-Hot Encoding. Columnas creadas (K-1 dummies): ['ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']


In [9]:
import numpy as np
from sklearn.preprocessing import RobustScaler

def transformar_y_escalar(df: pd.DataFrame, is_train: bool = True, scaler=None) -> tuple:
    df_proc = df.copy()
    
    # 1. Transformación Logarítmica (Variables fuertemente sesgadas según tu EDA)
    # Usamos np.log1p (log(1 + x)) para evitar el error de log(0) si hubiera algún valor en 0
    cols_sesgadas = ['population', 'total_rooms', 'total_bedrooms', 'households', 'median_income']
    
    for col in cols_sesgadas:
        if col in df_proc.columns:
            df_proc[f'{col}_log'] = np.log1p(df_proc[col])
            df_proc.drop(columns=[col], inplace=True) # Eliminamos la original
            
    # 2. One-Hot Encoding (Omitido aquí por brevedad, igual que antes)
    # ...
    
    # 3. Escalado con RobustScaler (Tu excelente sugerencia)
    cols_to_scale = [col for col in df_proc.columns if 'ocean_' not in col and col != 'median_house_value']
    
    if is_train:
        scaler = RobustScaler()
        df_proc[cols_to_scale] = scaler.fit_transform(df_proc[cols_to_scale])
    else:
        df_proc[cols_to_scale] = scaler.transform(df_proc[cols_to_scale])
        
    print(f"Transformación y escalado completados. Columnas procesadas: {cols_to_scale}")
        
    return df_proc, scaler

# Ejecución
df_final, fitted_scaler = transformar_y_escalar(df_step4, is_train=True)

Transformación y escalado completados. Columnas procesadas: ['longitude', 'latitude', 'housing_median_age', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'dist_to_LA', 'dist_to_SF', 'rooms_per_population', 'population_log', 'total_rooms_log', 'total_bedrooms_log', 'households_log', 'median_income_log']
